# Choosing a decoder

This notebook configures a decoder specification once, then uses it with a matrix, a detector error model, code-capacity sampling, and Sinter.
It ends with a small custom decoder passed directly as `decoder=`, followed by routing edge cases.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.


In [1]:
%%capture
%pip install qldpc

In [2]:
import galois
import numpy as np
import sinter
import stim

from qldpc import codes, decoders
from qldpc.codes import code_capacity

## Configure once, build for a matrix

Helpers under `qldpc.decoders` return reusable decoder specifications.
A specification stores typed options but is not bound to a particular matrix or detector error model until it is built.
See the [decoder guide](https://qldpc.readthedocs.io/decoders.html) for the available algorithms and their options.

In [3]:
spec = decoders.bp_osd(osd_method="OSD_CS", osd_order=2)
spec

decoders.bp_osd(osd_method='OSD_CS', osd_order=2)

The parity-check matrix determines how physical errors produce syndromes.
An **error decoder** infers a physical error from a syndrome with `decode_errors`.
Build one for that matrix, then check that the inferred error reproduces the syndrome:

In [4]:
code = codes.HammingCode(3)
error = code.field.Zeros(len(code))
error[2] = 1
syndrome = code.matrix @ error
inferred = spec.build(code.matrix).decode_errors(syndrome)
assert np.array_equal(code.matrix @ code.field(inferred), syndrome)
print("syndrome:", syndrome)
print("inferred error:", inferred)

syndrome: [0 1 1]
inferred error: [0 0 1 0 0 0 0]


## Predict observables from a detector error model

A detector error model (DEM) supplies error probabilities and observable targets.
An **observable decoder** predicts observable flips from detection events with `decode_observables`.
`build_observable_decoder(dem)` requests observable predictions: a lookup specification predicts the most likely observable class natively, whereas BP+OSD infers errors and computes corresponding observable flips.
For the detection events below, the most likely individual error flips the observable, but the combined probability of no-flip errors of weight at most two is greater.

In [5]:
dem = stim.DetectorErrorModel("""
    error(0.08) D0 D1
    error(0.25) D0
    error(0.10) D1
    error(0.10) D0 L0
    error(0.25) D1 L0
""")
events = np.array([1, 1], dtype=int)
native_spec = decoders.lookup(max_weight=2)
projected_spec = decoders.bp_osd()
for candidate in (native_spec, projected_spec):
    prediction = candidate.build_observable_decoder(dem).decode_observables(events)
    print(f"{candidate}: native={candidate.predicts_observables_natively}, flips={prediction}")

decoders.lookup(max_weight=2): native=True, flips=[0]
decoders.bp_osd(): native=False, flips=[1]


## Estimate a code-capacity logical error rate

Pass a decoder specification as `decoder=`; the estimator builds it for the code and counts incorrect logical predictions.

In [6]:
np.random.seed(3)
logical_error_rate = code.get_logical_error_rate_func(
    num_samples=100, max_error_rate=0.1, decoder=decoders.lookup(max_weight=1)
)
print("logical error rate and uncertainty:", logical_error_rate(0.01))

logical error rate and uncertainty: (0.002031041634940002, 0.0006342533188556292)


## Decode a circuit with Sinter

Sinter compiles a `SinterDecoder` for a circuit's DEM.
Pass a decoder specification as `decoder=`; a compiled decoder predicts observable flips for individual shots or batches.
Sinter counts logical failures when predicted observables disagree with sampled observables.

In [7]:
circuit = stim.Circuit.generated(
    "repetition_code:memory", distance=3, rounds=3, after_clifford_depolarization=0.1
)
sinter_decoder = decoders.SinterDecoder(decoder=decoders.mwpm())
compiled = sinter_decoder.compile_decoder_for_dem(circuit.detector_error_model())
sample = circuit.compile_detector_sampler(seed=3).sample(1)[0]
print("predicted observable flips:", compiled.decode_observables(sample))

(stat,) = sinter.collect(
    tasks=[sinter.Task(circuit=circuit, decoder="qldpc-mwpm")],
    custom_decoders={"qldpc-mwpm": sinter_decoder},
    num_workers=1,
    max_shots=100,
    print_progress=False,
)
print(f"Sampled logical errors: {stat.errors} of {stat.shots}")

predicted observable flips: [0]
Sampled logical errors: 5 of 100


## Add a small custom decoder

For a small binary code, the following decoder checks which matrix column matches a syndrome.
It corrects at most one error.
Pass the constructor itself as `decoder=`: code capacity supplies the matrix when it builds the decoder.
The [adding a decoder guide](https://qldpc.readthedocs.io/adding_decoders.html) covers reusable backends, batch decoding, and erasure support.

In [8]:
class SingleErrorDecoder(decoders.ErrorDecoder):
    def __init__(self, matrix: galois.FieldArray) -> None:
        if type(matrix).order != 2:
            raise ValueError("SingleErrorDecoder requires a binary matrix")
        self.columns = np.asarray(matrix, dtype=int).T

    def decode_errors(self, syndrome: np.ndarray) -> np.ndarray:
        matches = np.flatnonzero(np.all(self.columns == np.asarray(syndrome), axis=1))
        correction = np.zeros(len(self.columns), dtype=int)
        if len(matches):
            correction[matches[0]] = 1
        return correction


assert np.array_equal(SingleErrorDecoder(code.matrix).decode_errors(syndrome), error)
np.random.seed(3)
estimate = code.get_logical_error_rate_func(
    num_samples=100, max_error_rate=0.1, decoder=SingleErrorDecoder
)
print("custom constructor:", estimate(0.01))

custom constructor: (0.002031041634940002, 0.0006342533188556292)
